Imports & Setup:-
In this section, we import all required libraries and define the directory structure for the dataset. This ensures consistent file access and reproducibility across environments.

In [8]:


import os
import pandas as pd
import numpy as np

from tqdm import tqdm
import whisper
import language_tool_python

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
from sklearn.feature_extraction.text import TfidfVectorizer


In [2]:
ROOT_DIR = os.getcwd()

DATASET_DIR = os.path.join(ROOT_DIR, "dataset")

AUDIO_DIR = os.path.join(DATASET_DIR, "audios")
TRAIN_AUDIO_DIR = os.path.join(AUDIO_DIR, "train")
TEST_AUDIO_DIR = os.path.join(AUDIO_DIR, "test")

CSV_DIR = os.path.join(DATASET_DIR, "csvs")
TRAIN_CSV_PATH = os.path.join(CSV_DIR, "train.csv")
TEST_CSV_PATH = os.path.join(CSV_DIR, "test.csv")

train_df = pd.read_csv(TRAIN_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()


Train shape: (409, 2)
Test shape: (197, 1)


,filename,label
0,audio_173,3.0
1,audio_138,3.0
2,audio_127,2.0
3,audio_95,2.0
4,audio_73,3.5


Data Preprocessing (Audio -> Text)

In this step, we convert raw speech audio into text using a pretrained Whisper speech-to-text model. This allows us to transform unstructured audio data into textual data suitable for downstream grammar analysis and feature extraction.

In [3]:
sample_file = train_df.iloc[0]['filename']
sample_audio_path = os.path.join(TRAIN_AUDIO_DIR, sample_file + ".wav")

print("Sample audio file:", sample_audio_path)
print("Exists?", os.path.exists(sample_audio_path))
model = whisper.load_model("base")

result = model.transcribe(sample_audio_path, fp16=False)
print(result["text"])
transcriptions = []

for idx, row in tqdm(train_df.iterrows(), total=len(train_df)):
    filename = row["filename"]
    score = row["label"]

    audio_path = os.path.join(TRAIN_AUDIO_DIR, filename + ".wav")

    if not os.path.exists(audio_path):
        continue

    result = model.transcribe(audio_path, fp16=False)
    text = result["text"].strip()

    transcriptions.append({
        "filename": filename,
        "transcription": text,
        "grammar_score": score
    })

train_text_df = pd.DataFrame(transcriptions)

OUTPUT_DIR = os.path.join(DATASET_DIR, "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_TEXT_PATH = os.path.join(OUTPUT_DIR, "train_transcriptions.csv")
train_text_df.to_csv(TRAIN_TEXT_PATH, index=False)

train_text_df.head()


Sample audio file: C:\Users\Asus\PycharmProjects\JupyterProject\dataset\audios\train\audio_173.wav
Exists? True
 My favorite place to visit will be Japan, because I'm really interested in their culture. I'm really into all Japanese stuff like anime, their history, etc. So that's why my favorite place to visit. Starting the food that I would like to try, I would like to try the dishes. I would like to try the cakes that they sell that have different flavors mainly for fruits. And the season that I would like to go is during the cherry seasons when only trees have like those pink leaves. So I think that's really cool.


100%|██████████| 409/409 [54:48<00:00,  8.04s/it]  


,filename,transcription,grammar_score
0,audio_173,"My favorite place to visit will be Japan, beca...",3.0
1,audio_138,I love to reading on my hobby such reading. Em...,3.0
2,audio_127,My favorite place to visit is Mullah Itis near...,2.0
3,audio_95,I am going to tell about my hobby. And my hobb...,2.0
4,audio_73,This is a tough one. So my bestie of my life i...,3.5


Feature Engineering (Without NLP)

Here, we extract simple, interpretable grammar-related features such as grammar error counts, text length, and word statistics. These features serve as a baseline representation of grammatical quality without using advanced NLP techniques.

In [4]:
df = pd.read_csv(TRAIN_TEXT_PATH)

tool = language_tool_python.LanguageTool('en-US')
def extract_grammar_features(text):
    matches = tool.check(text)

    words = text.split()
    num_words = len(words)
    text_length = len(text)

    # Sentence approximation (simple but effective)
    sentences = [s for s in text.split('.') if len(s.strip()) > 0]
    num_sentences = max(len(sentences), 1)

    # Vocabulary diversity
    unique_words = set(words)

    return {
        "num_grammar_errors": len(matches),
        "text_length": text_length,
        "num_words": num_words,
        "avg_word_length": sum(len(w) for w in words) / max(num_words, 1),

        "errors_per_word": len(matches) / max(num_words, 1),
        "errors_per_sentence": len(matches) / num_sentences,

        "num_sentences": num_sentences,
        "avg_sentence_length": num_words / num_sentences,

        "unique_word_ratio": len(unique_words) / max(num_words, 1),

        "comma_count": text.count(','),
        "period_count": text.count('.')
    }


features = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    feats = extract_grammar_features(row["transcription"])
    feats["score"] = row["grammar_score"]
    features.append(feats)

feature_df = pd.DataFrame(features)
feature_df.head()
X = feature_df.drop(columns=["score"])
y = feature_df["score"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


100%|██████████| 409/409 [00:37<00:00, 10.85it/s]


Model Training & Evaluation (Without NLP)

Using the handcrafted grammar features, we train and evaluate three regression models: Linear Regression, Random Forest, and Gradient Boosting. Performance is measured using RMSE and Pearson correlation to assess both error magnitude and ranking consistency.

In [5]:
def evaluate_model(model, X_train, y_train, X_val, y_val, name):
    model.fit(X_train, y_train)

    train_preds = model.predict(X_train)
    val_preds = model.predict(X_val)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    pearson_corr, _ = pearsonr(y_val, val_preds)

    print(f"\n{name}")
    print(f"Training RMSE: {train_rmse:.4f}")
    print(f"Validation RMSE: {val_rmse:.4f}")
    print(f"Pearson Correlation: {pearson_corr:.4f}")
evaluate_model(LinearRegression(), X_train, y_train, X_val, y_val, "Linear Regression")
evaluate_model(RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
               X_train, y_train, X_val, y_val, "Random Forest")
evaluate_model(GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3),
               X_train, y_train, X_val, y_val, "Gradient Boosting")



Linear Regression
Training RMSE: 0.6790
Validation RMSE: 0.7093
Pearson Correlation: 0.3550

Random Forest
Training RMSE: 0.4916
Validation RMSE: 0.7215
Pearson Correlation: 0.3526

Gradient Boosting
Training RMSE: 0.3535
Validation RMSE: 0.7497
Pearson Correlation: 0.3154


Feature Engineering (With NLP)

This section enhances the feature space using NLP-based representations. TF-IDF is applied to capture lexical patterns and n-gram information from transcriptions, which complements grammar-based features.

In [6]:
tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),
    stop_words="english"
)

tfidf_features = tfidf.fit_transform(df["transcription"])

tfidf_df = pd.DataFrame(
    tfidf_features.toarray(),
    columns=tfidf.get_feature_names_out()
)

grammar_df = feature_df.drop(columns=["score"])

X = pd.concat([grammar_df, tfidf_df], axis=1)
y = df["grammar_score"]

print("Final feature shape:", X.shape)


Final feature shape: (409, 311)


Model Training & Evaluation (With NLP)

Finally, we retrain and evaluate the same regression models using the enriched feature set that includes NLP features. The results are compared against non-NLP models to assess performance gains.

In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)
evaluate_model(LinearRegression(), X_train, y_train, X_val, y_val, "Linear Regression + NLP")
evaluate_model(RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42, n_jobs=-1),
               X_train, y_train, X_val, y_val, "Random Forest + NLP")
evaluate_model(GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3),
               X_train, y_train, X_val, y_val, "Gradient Boosting + NLP")



Linear Regression + NLP
Training RMSE: 0.0744
Validation RMSE: 4.4020
Pearson Correlation: 0.0745

Random Forest + NLP
Training RMSE: 0.4861
Validation RMSE: 0.6849
Pearson Correlation: 0.4326

Gradient Boosting + NLP
Training RMSE: 0.2865
Validation RMSE: 0.6911
Pearson Correlation: 0.4206


In [10]:
x_full = X
y_full = y
rf = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42, n_jobs=-1)
rf.fit(x_full, y_full)

test_transcriptions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    filename = row["filename"]
    audio_path = os.path.join(TEST_AUDIO_DIR, filename + ".wav")

    if not os.path.exists(audio_path):
        print(f"Missing test file: {audio_path}")
        continue

    result = model.transcribe(audio_path, fp16=False)
    text = result["text"].strip()

    test_transcriptions.append({
        "filename": filename,
        "transcription": text
    })

test_text_df = pd.DataFrame(test_transcriptions)

print("Test transcriptions shape:", test_text_df.shape)

test_grammar_features = []

for _, row in tqdm(test_text_df.iterrows(), total=len(test_text_df)):
    feats = extract_grammar_features(row["transcription"])
    test_grammar_features.append(feats)

test_grammar_df = pd.DataFrame(test_grammar_features)

test_tfidf_features = tfidf.transform(test_text_df["transcription"])
test_tfidf_df = pd.DataFrame(
    test_tfidf_features.toarray(),
    columns=tfidf.get_feature_names_out()
)

X_test_final = pd.concat([test_grammar_df, test_tfidf_df], axis=1)

print("Final test feature shape:", X_test_final.shape)
test_predictions = rf.predict(X_test_final)

submission_df = pd.DataFrame({
    "filename": test_text_df["filename"],
    "label": test_predictions
})

SUBMISSION_PATH = os.path.join(DATASET_DIR, "rf_nlp_predictions.csv")
submission_df.to_csv(SUBMISSION_PATH, index=False)

print("Predictions saved to:", SUBMISSION_PATH)
submission_df.head()


100%|██████████| 197/197 [23:17<00:00,  7.09s/it]


Test transcriptions shape: (197, 2)


100%|██████████| 197/197 [00:17<00:00, 11.21it/s]


Final test feature shape: (197, 311)
Predictions saved to: C:\Users\Asus\PycharmProjects\JupyterProject\dataset\rf_nlp_predictions.csv


,filename,label
0,audio_141,2.727804
1,audio_114,2.887393
2,audio_17,2.681516
3,audio_76,4.012909
4,audio_156,2.861900
